In [1]:
import torchvision
import torchvision.transforms as transforms
# from torchvision.models import resnet50, ResNet50_Weight
from data_utils.cub2011 import Cub2011
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pytorch_influence_functions as ptif
import pandas as pd
import ast
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [2]:
stratify="0.5"
data_path = "/tmp2/IF_manipulation_files/"
dataset = "cifar10"

In [3]:
loaded_df_test = pd.read_csv(f'{data_path}resnet50_features/test_val_{stratify}/{dataset}_val_features_labels.csv')

# Convert DataFrame columns back to tensors
test_feat_loaded_df = loaded_df_test['features'].apply(ast.literal_eval) 
test_feat_loaded_tensor = torch.tensor(test_feat_loaded_df.tolist())

test_labels_loaded_tensor = torch.tensor(loaded_df_test['gt_labels'].tolist())

# Reshape tensors to their original sizes
n_rows, n_cols = loaded_df_test.shape[0], len(loaded_df_test.iloc[0].features.split(','))
test_x_feat_tensor = test_feat_loaded_tensor.view(n_rows, n_cols)
test_labels_tensor = test_labels_loaded_tensor.view(n_rows)

In [4]:
unique_values = torch.unique(test_labels_tensor)
num_classes = unique_values.numel()
print(num_classes)

10


In [5]:
# Define your model
class LinearClassifier(nn.Module):
    def __init__(self, input_size=2048, num_classes=num_classes):
        super(LinearClassifier, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

In [6]:
mean_std_df = pd.read_csv(f'{data_path}resnet50_features/test_val_{stratify}/{dataset}_train_mean_std.csv')
mean= torch.tensor(mean_std_df['Mean'].values)
std= torch.tensor(mean_std_df['Std'].values)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64  # You can adjust the batch size as needed
model_path = f'{data_path}resnet50_features/test_val_{stratify}/models/linear_classifier_{dataset}.pt'

input_size = test_x_feat_tensor.shape[1]
loaded_model = LinearClassifier()
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.to(device)
loaded_model.eval()

LinearClassifier(
  (linear): Linear(in_features=2048, out_features=10, bias=True)
)

In [8]:
# After training, you can use the model for predictions
# For example, if you have a test feature tensor called test_features:
# Move test data to GPU if available

test_std_features = (test_x_feat_tensor - mean)/std
test_std_features = test_std_features.to(device)
test_labels = test_labels_tensor.to(device)
#test_dataset = TensorDataset(test_std_features, test_labels)
#test_dataloader_noshuffle = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

test_std_features = test_std_features.to(torch.float32)  # or torch.float, depending on your model's parameters data type

# Ensure that test_labels_tensor is also converted to the same device and data type
test_labels = test_labels_tensor.to(device, dtype=torch.float32)

test_outputs = loaded_model(test_std_features)
predicted_labels = torch.argmax(test_outputs, dim=1)

# Calculate test accuracy
test_accuracy = torch.sum(predicted_labels == test_labels).item() / len(test_labels)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 89.12%
